# 2. Run SAM 3 on tiles (GPU)

SAM 3 (Meta) segments every object matching a short text prompt. This project runs it through SamGeo 1.4.2 at a pinned SAM 3 commit, tile by tile, and turns each mask into a georeferenced polygon with its confidence score.

It needs an NVIDIA GPU. Lewisville ran on a 6 GB RTX 4050 laptop GPU under WSL2 (Ubuntu 24.04), and this notebook was executed in that environment on the synthetic demo. Without a GPU or the checkpoint, it prints the plan and skips the model steps.

## Environment

```bash
# In WSL Ubuntu (or Linux), from the repository root
bash scripts/setup_sam3.sh   # torch 2.10.0 (CUDA 12.8), SAM 3 at the pinned commit, SamGeo 1.4.2
```

The checkpoint is gated: request access to `facebook/sam3` on Hugging Face, sign in with your own account, and place `sam3.pt` in `models/`. The next cells check for a GPU and the checkpoint and verify its published SHA-256.

In [1]:
import json
import shutil
from pathlib import Path

from nearmap_buildings.cli import main as _nbf

# Run from the repository root or from notebooks/. Generated files go to notebook_runs/, which git ignores.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WORK = REPO / "notebook_runs" / "02_run_sam3"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)


def nbf(*args):
    """Run one nbf command in this process; stop the notebook if it fails."""
    code = _nbf([str(a) for a in args])
    if code:
        raise RuntimeError(f"nbf {args[0]} failed with exit code {code}")

CHECKPOINT = REPO / "models" / "sam3.pt"
try:
    import torch
    GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    GPU = None
RUN_SAM3 = bool(GPU) and CHECKPOINT.is_file()
print("GPU:", GPU, "| checkpoint present:", CHECKPOINT.is_file(), "| running SAM 3:", RUN_SAM3)

GPU: NVIDIA GeForce RTX 4050 Laptop GPU | checkpoint present: True | running SAM 3: True


In [2]:
from nearmap_buildings.common import sha256_file

EXPECTED_SHA256 = "9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e"
if CHECKPOINT.is_file():
    print("checkpoint matches the published SHA-256:", sha256_file(CHECKPOINT) == EXPECTED_SHA256)

checkpoint matches the published SHA-256: True


In [3]:
DEMO = WORK / "demo"
nbf("demo", "--output", DEMO)
sorted(p.name for p in DEMO.iterdir())

nbf("tile", DEMO / "imagery.tif", "--output", WORK / "tiles_256", "--tile-size", 256, "--overlap", 32)
MANIFEST = WORK / "tiles_256" / "manifest.json"

<repo>/notebook_runs/02_run_sam3/demo
Prepared 21 tiles: <repo>/notebook_runs/02_run_sam3/tiles_256/manifest.json


## Plan first

Without `--execute`, `nbf infer` checks the manifest, the tiles and the settings, prints the plan and loads nothing. The settings are the ones frozen for Lewisville: the prompt "building" and a low confidence floor of 0.3, so any cut-off can be applied later from the stored scores without running the model again.

In [4]:
RUN = WORK / "sam3_building"
settings = ["--manifest", MANIFEST, "--method", "text", "--text", "building", "--confidence", 0.3, "--output", RUN]
nbf("infer", *settings)

{
  "dry_run": true,
  "method": "text",
  "text": "building",
  "tile_count": 21,
  "manifest": "<repo>/notebook_runs/02_run_sam3/tiles_256/manifest.json",
  "manifest_sha256": "15c0565a9d4f7c560b14ef200cbfd080e79992d5afecea5de0beea9eee618cbc",
  "prompts": null,
  "prompts_sha256": null,
  "checkpoint": "facebook/sam3 (Hugging Face)",
  "confidence": 0.3,
  "min_pixels": 16,
  "device": "cuda",
  "limit": null
}


## Run

Each tile gets its own GeoPackage and a receipt holding the tile's hash, its instance count and the time it took. If a run stops, the same command with `--resume` skips every tile whose receipt and output still match.

SAM 3 finds what the prompt names. The demo's flat beige rectangles on flat green don't look like buildings, so the frozen Lewisville prompt, "building", may find nothing here. The cell also runs a prompt that names what the demo image shows, "rectangle", and the rest of the notebook uses that run. On real imagery the prompt was one of the tuned settings: "building" beat "house" and "roof" in Old Town.

In [5]:
def run_sam3(prompt):
    out = WORK / f"sam3_{prompt}"
    nbf("infer", "--manifest", MANIFEST, "--method", "text", "--text", prompt, "--confidence", 0.3,
        "--output", out, "--checkpoint", CHECKPOINT, "--execute")
    run = json.loads((out / "run.json").read_text())
    seconds = sorted(t["seconds"] for t in run["tiles"])
    print(f"prompt {prompt!r}: {run['status']}, {len(run['tiles'])} tiles, {sum(t['instances'] for t in run['tiles'])} instances, "
          f"{run['elapsed_seconds']:.0f} s including model loading, median {seconds[len(seconds) // 2]:.2f} s per tile")
    return out


if RUN_SAM3:
    run_sam3("building")
    RUN = run_sam3("rectangle")
else:
    print("Skipped: needs a CUDA GPU and models/sam3.pt.")

Using cuda device and meta backend


No objects found. Please try a different prompt.
r00000000_c00000000: complete, 0 instances


No objects found. Please try a different prompt.
r00000000_c00000224: complete, 0 instances


No objects found. Please try a different prompt.
r00000000_c00000448: complete, 0 instances


No objects found. Please try a different prompt.
r00000000_c00000672: complete, 0 instances


...

Found one object.
r00000256_c00000448: complete, 1 instances


No objects found. Please try a different prompt.
r00000256_c00000672: complete, 0 instances


No objects found. Please try a different prompt.
r00000256_c00000896: complete, 0 instances


No objects found. Please try a different prompt.
r00000256_c00001120: complete, 0 instances


No objects found. Please try a different prompt.
r00000256_c00001280: complete, 0 instances
prompt 'rectangle': complete, 21 tiles, 9 instances, 19 s including model loading, median 0.53 s per tile


## Apply a cut-off and clean

Raw instances overlap where tiles overlap, and some are cut by a tile edge. Cleanup repairs geometry, drops pieces under 4 m² and suppresses duplicates, keeping a complete detection before one cut by a tile edge. The cut-off is applied first, from the stored scores; 0.7 is the value frozen for Lewisville.

In [6]:
if RUN_SAM3:
    from nearmap_buildings.postprocess import clean_polygons, read_polygon_inputs, write_layers

    raw, _ = read_polygon_inputs(RUN / "raw")
    print({f">= {c}": int((raw.score >= c).sum()) for c in (0.3, 0.5, 0.7, 0.9)})
    result = clean_polygons(raw[raw.score >= 0.7], crs="EPSG:32614", min_area=4)
    write_layers(WORK / "sam3_cleaned.gpkg", {"cleaned": result.cleaned, "removed_audit": result.removed}, overwrite=True)
    print(len(result.cleaned), "outlines kept;", result.removed.removed_reason.value_counts().to_dict())

{'>= 0.3': 9, '>= 0.5': 6, '>= 0.7': 5, '>= 0.9': 0}


3 outlines kept; {'duplicate_overlap': 2}


## Check against the synthetic reference

The demo's reference outlines are exact by construction, so this is a true holdout check of the plumbing, not evidence about real imagery.

In [7]:
if RUN_SAM3:
    nbf("evaluate", "--predictions", WORK / "sam3_cleaned.gpkg", "--predictions-layer", "cleaned",
        "--reference", DEMO / "reference.gpkg", "--aoi", DEMO / "aoi.geojson", "--metric-crs", "EPSG:32614",
        "--output-json", WORK / "sam3_evaluation.json", "--independent-holdout")
    report = json.loads((WORK / "sam3_evaluation.json").read_text())
    print({k: report[k] for k in ("true_positives", "false_positives", "false_negatives", "precision", "recall", "f1")})

{"true_positives": 2, "false_positives": 1, "false_negatives": 7, "precision": 0.6666666666666666, "recall": 0.2222222222222222, "f1": 0.3333333333333333}
{'true_positives': 2, 'false_positives': 1, 'false_negatives': 7, 'precision': 0.6666666666666666, 'recall': 0.2222222222222222, 'f1': 0.3333333333333333}


## The Lewisville run

The city-wide run used the same settings on the 1,536 px tiles, from WSL, with the tiles read from the Windows drive:

```bash
cd ~/nearmap-building-footprints && source .venv-sam3/bin/activate
out=~/runs/lewisville/city/sam3_building_1536
resume=""; [[ -f "$out/run.json" ]] && resume="--resume"
nbf infer --manifest tiles_1536/manifest.json --method text --text building \
  --confidence 0.3 --checkpoint models/sam3.pt --output "$out" --execute $resume
```

| | Lewisville |
|---|---|
| Tiles | 6,538 of 1,536 px |
| Time | 2 h 41 min; median 1.08 s of model time per tile |
| GPU memory | up to 5.8 GB of 6 GB |
| Outlines | 84,293 raw at ≥ 0.3, 50,716 at ≥ 0.7, 38,911 after cleanup |

Next: [3. Run Esri](03_run_esri.ipynb).